# DJEF Chef — from the notebooks to a working Algerian agent 🇩🇿

**by Kassouar Fatima** — builds the same agent the Streamlit app and LangGraph Studio run, one piece at a time, reusing the four earlier notebooks: `1_1` model calls · `1_2` tools + web search · `1_3` memory · `1_4` multimodal messages.

- **Run:** double-click `run_notebook.cmd` in this `djef_chef` folder → *Run → Run All Cells*.
- **Needs:** the repo's `.env` at the project root (`GEMINI_API_KEY`, `TAVILY_API_KEY`).


## Setup — locate the project and check services

> **Kernel check:** the kernel is already pinned to **Python (djef)** in this file.
> The cell finds the project root by itself, wherever the notebook was opened from.

> **No scary errors:** every Gemini call in this notebook retries by itself
> when Google is briefly busy (429 / 5xx). If a call still fails, the cell
> prints one friendly line — never a red traceback.

> **Free-tier limits (normal, not bugs):** the free Gemini tier allows ~10
> calls per minute and ~20 calls per model per day. A full "Run All" uses
> about 8 calls, so budget about 2 runs per day per model.
> - A "temporarily busy" line after a burst → wait ~30–60 s and run that cell
>   again — it then works.
> - A "daily quota is used up" line → that model's 20/day bucket is done for
>   today: it resets at 00:00 Pacific time, or set `CHEF_MODEL` in `.env` to
>   another current flash model (e.g. `gemini-3.6-flash`) for a fresh bucket.
> - `404 NOT_FOUND` → the model is retired for new accounts; keep `CHEF_MODEL`
>   on a current model (default **`gemini-3.6-flash`**) in `.env` and restart
>   the kernel (Kernel → Restart).


In [1]:
import sys
from pathlib import Path

def _find_project_root() -> Path | None:
    """Locate the project root that contains notebook/chef_service.py."""
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "notebook" / "chef_service.py").exists():
            return folder
    return None

PROJECT_ROOT = _find_project_root()
if PROJECT_ROOT is None:
    raise SystemExit(
        "Could not find 'notebook/chef_service.py' near "
        f"{Path.cwd()!s}. Start Jupyter from the project folder and retry."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))  # this project's 'notebook' package

from notebook import chef_service as chef  # noqa: E402

print("Services:", chef.service_status())
print("Python :", sys.executable)
print("Root   :", PROJECT_ROOT)


Services: {'gemini_configured': True, 'tavily_configured': True, 'tracing_enabled': False, 'model': 'gemini-3.6-flash'}
Python : C:\Users\Kassouar Fatima\Downloads\courses\Personal Algerian Chef\.venv\Scripts\python.exe
Root   : C:\Users\Kassouar Fatima\Downloads\courses\Personal Algerian Chef


## 1 · The model (notebook `1_1`)

`ChatGoogleGenerativeAI` is the provider-specific twin of
`init_chat_model("google_genai:...")`. A model answer is an *object*: the text
lives in `response.text`, and `usage_metadata` tells you the token bill.

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model=chef.service_status()["model"],
    google_api_key=chef.get_gemini_api_key(),
    temperature=0.35,  # the four dials: temperature, max_tokens, timeout, max_retries
)

try:
    response = chef.invoke_with_retry(model.invoke,
        "Say Salaam in one short Algerian phrase, then name the model that answered."
    )
    print(response.text)
    print("Tokens used:", response.usage_metadata)
except Exception as _gem_exc:
    chef.handle_gemini_error(_gem_exc)


C:\Users\Kassouar Fatima\Downloads\courses\Personal Algerian Chef\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Salam alaykoum, washrak? 

I am **Gemini**.
Tokens used: {'input_tokens': 16, 'output_tokens': 381, 'total_tokens': 397, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 364}}


## 2 · Tools (notebook `1_2`)

The `@tool` decorator turns a Python function into an agent tool. The agent only
sees the **name**, the **description**, and the **args schema** — so good
descriptions matter. `match_leftover_recipes` scores the curated Algerian menu.

In [3]:
from notebook.chef_service import match_leftover_recipes

print("tool name :", match_leftover_recipes.name)
print("tool args :", match_leftover_recipes.args)
print()
print(
    match_leftover_recipes.invoke(
        {
            "ingredients": "lentils, tomatoes, onion",
            "dietary_preferences": "vegetarian",
            "allergies": "",
        }
    )
)

tool name : match_leftover_recipes
tool args : {'ingredients': {'title': 'Ingredients', 'type': 'string'}, 'dietary_preferences': {'default': '', 'title': 'Dietary Preferences', 'type': 'string'}, 'allergies': {'default': '', 'title': 'Allergies', 'type': 'string'}}

{"matches": [{"title": "Harira", "translation": "حريرة", "category": "Algerian soup", "match_percent": 100, "servings": 2, "use_now": ["lentil", "tomato", "onion"], "still_missing": [], "you_also_need": ["olive oil", "cumin", "paprika", "salt", "water"], "optional_add_ins": ["chickpea", "onion", "carrot", "garlic"], "method": ["Soften onion, garlic, carrot, and tomato in olive oil.", "Add lentils, optional chickpeas, cumin, paprika, and water.", "Simmer until tender, stirring so the base does not stick.", "Season to taste and serve with bread and lemon."], "safety_notes": []}, {"title": "Chakhchoukha", "translation": "شخشوخة", "category": "Traditional feast", "match_percent": 100, "servings": 2, "use_now": ["tomato", "onio

## 3 · The agent = model + tools + system prompt (notebooks `1_1` & `1_2`)

`create_agent` wraps the model with the tools. Run the graph, then **inspect the
message list** — the middle message is the *action* (an `AIMessage` with
`tool_calls`), followed by the *observation* (`ToolMessage`), then the answer.

In [4]:
from langgraph.checkpoint.memory import MemorySaver

from notebook.chef_service import build_chef_graph, conversation_config

graph = build_chef_graph(checkpointer=MemorySaver())
config = conversation_config("notebook-1")

try:
    result = chef.invoke_with_retry(graph.invoke,
        {
            "messages": [
                {
                    "role": "user",
                    "content": "I have chickpeas, tomatoes, onion, and cumin. What can I make?",
                }
            ]
        },
        config,
    )
    for i, message in enumerate(result["messages"]):
        calls = getattr(message, "tool_calls", None)
        print(i, type(message).__name__, [c["name"] for c in calls] if calls else "")
    print("\nANSWER:", result["messages"][-1].content)
except Exception as _gem_exc:
    chef.handle_gemini_error(_gem_exc)


C:\Users\Kassouar Fatima\Downloads\courses\Personal Algerian Chef\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
C:\Users\Kassouar Fatima\Downloads\courses\Personal Algerian Chef\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


0 HumanMessage 
1 AIMessage ['match_leftover_recipes']
2 ToolMessage 
3 AIMessage 

ANSWER: [{'type': 'text', 'text': "Marhaba! With chickpeas, tomatoes, onion, and cumin, you have the foundation for a comforting Algerian classic: **Tajine el Homs** (Chickpea Tagine / طاجين الحمص) or a quick version of Biskra's famous spicy chickpea dish, **Dubbara** (دوبارة). \n\nHere is my top recommendation:\n\n---\n\n### Primary Recommendation: **Tajine el Homs (طاجين الحمص)**\nA warm, fragrant Algerian chickpea stew in a cumin-infused tomato sauce. It comes together quickly and goes wonderfully with fresh crusty bread (*kesra* or baguette).\n\n* **Why it fits:** It highlights every ingredient you have on hand and turns simple staples into a deeply flavorful, satisfying meal.\n* **Ingredients you have:** Chickpeas, tomatoes, onion, cumin.\n* **Pantry basics needed:** Olive oil, salt, a pinch of paprika (or chili/harissa if you like heat), garlic (optional), and water.\n* **Short Steps:**\n  1. **Sa

## 4 · Streaming and memory (notebooks `1_1` & `1_3`)

`stream_mode="messages"` yields token chunks; the checkpointer snapshots the
state after every turn under the thread id, so the next turn still knows the
ingredients without repeating them.

In [5]:
try:
    for token, _meta in graph.stream(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "Make the tagine vegan for four people with exact quantities.",
                }
            ]
        },
        config,
        stream_mode="messages",
    ):
        if token.text:
            print(token.text, end="", flush=True)
except Exception as _g_exc:
    print()
    chef.handle_gemini_error(_g_exc)

snapshot = graph.get_state(config)
print("\nMessages saved for this thread:", len(snapshot.values["messages"]))


C:\Users\Kassouar Fatima\Downloads\courses\Personal Algerian Chef\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{"query": "tajine el homs algerien vegan recipe proportions", "answer": "A vegan tajine el homs typically includes carrots, potatoes, chickpeas, zucchini, tomatoes, and spices like cumin and coriander. It's naturally gluten-free and rich in fiber. The dish is slow-cooked to enhance flavors.", "sources": [{"title": "Tajine vegan", "url": "https://www.fourchette-et-bikini.fr/recettes/tajine-vegan-39905.html", "snippet": "Title: Tajine vegan\nLe tajine vegan invite voyage et partage au cœur de la cuisine maghrébine. Carottes, pommes de terre, pois chiches, courgettes et tomates forment une alliance haute en couleurs et en saveurs. Ce plat réconfortant souligne le meilleur des légumes de saison, parfumé d'épices douces et de coriandre fraîche. Épluchez les carottes et les pommes de terre, puis coupez-les en morceaux réguliers. Rincez les courgettes, les tomates et le bouquet de coriandre, puis hachez grossièrement la coriandre. Ajoutez l’ail, le cumin, le curcuma et remuez pour libérer les

C:\Users\Kassouar Fatima\Downloads\courses\Personal Algerian Chef\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Here is the exact recipe for **Vegan Tajine el Homs (طاجين الحمص)**, scaled generously for **4 people**. Since traditional Tajine el Homs relies on legumes, fresh vegetables, olive oil, and warm spices, it is naturally vegan and full of rich flavor.

---

### **Vegan Tajine el Homs (طاجين الحمص)**
**Servings:** 4  
**Prep time:** 10 mins | **Cook time:** 25 mins

---

### **Ingredients**

* **Chickpeas:** 500 g cooked chickpeas (about two 400 g cans, rinsed and drained; or 250 g dry chickpeas, soaked overnight and cooked soft)
* **Tomatoes:** 400 g ripe tomatoes (3–4 medium tomatoes, grated or finely diced)
* **Onions:** 2 medium yellow onions (approx. 250 g total), finely chopped or grated
* **Garlic:** 3 cloves, minced or finely grated
* **Olive Oil:** 45 ml (3 tbsp) extra virgin olive oil
* **Spices:**
  * 1.5 tsp ground cumin (plus extra for garnish)
  * 1 tsp sweet paprika (or 1 tsp harissa paste if you prefer heat)
  * 1 tsp salt (adjust to taste)
  * ¼ tsp ground black pepper
* 

## 5 · The thread id is the memory key (notebook `1_3`)

A different `thread_id` means a different conversation. Ask the same question
on a fresh thread: the agent no longer remembers the fridge content.

In [6]:
try:
    reply = chef.invoke_with_retry(graph.invoke,
        {"messages": [{"role": "user", "content": "What did I say I had in the fridge?"}]}, config
    )["messages"][-1].content
    print("Same thread :", reply)
except Exception as _gem_exc:
    chef.handle_gemini_error(_gem_exc)

other = conversation_config("notebook-2")
try:
    reply = chef.invoke_with_retry(graph.invoke,
        {"messages": [{"role": "user", "content": "What did I say I had in the fridge?"}]}, other
    )["messages"][-1].content
    print("Other thread:", reply)
except Exception as _gem_exc:
    chef.handle_gemini_error(_gem_exc)


C:\Users\Kassouar Fatima\Downloads\courses\Personal Algerian Chef\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Same thread : [{'type': 'text', 'text': 'You mentioned having **chickpeas, tomatoes, onion, and cumin**.', 'extras': {'signature': 'EtIECs8EAWkUfRNlu7PkiQauwCDI2T16eYdBAPXhf77YshLJYFBKND13+o7TliPl779ugp7+GRY5EFRmKH9Wty+IcWrowSIbKA6elVDdr3Gpk7PoRAY/WkkXIWKITy9QchnfSIyrDTmkBy9hX0qdQacmcOB+XMN9w6vBA2JOpDQFKfF8pTuVEHvsiqiyGs2zrlOqPvDcRuVoFiFd2YOCAzot2xQsoEHEiHznjHR2k5Bx+83DRkgYAAbUtLnQRBnChKV1of6nSaallr44U8SnG2HG+6GJXE0LW9l4w0zK87sNCph/mkoA+ewVt7PXOrOC9Ipjgz29tkCezelqaBFCY0jYIXfg71gseX485vlvQz50Tn01aE+m2sSN6NGY6Y7q6WrMQCVReacunqRJ7dhQlb05C4gJGJ4c1gmLfo/ndkeGI6d8RqURj1qvPULaDmTHgoNNWySdWqdvIpeujr3O54/DoXmnI0+MFLj9a45JCrcp9xO0AB0kdJj+Ir4N87w63fU7Whob2xQyOSOc6d8o3KUvJu+RcdfB0Kv9ERbHILRtb9uGNd3WLGkodFbmyy0OoYjx2nXcREhp8zEPpD8L6l61Bva6LpGLrVLnYVFVGU5vFtab9nyttwijeB0P1065IstDN3rLKUMNFNyg5LvVi0hoDRGuk+S0gXcfv77gkYCr6h1Dho/oElsswkZeUjsjHS2iQQ6ebXUGRqwG+xaTmDx/mEPhSn1XEw/gW1GUUNG1rMmLp8j7252cCgn8ifp9mpQEgmyEH8VUb2gHAVzu31J+6jIv'}}]


C:\Users\Kassouar Fatima\Downloads\courses\Personal Algerian Chef\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Other thread: [{'type': 'text', 'text': "You haven't mentioned what's in your fridge yet! \n\nTell me what ingredients or leftovers you have on hand—even if it's just a few vegetables, eggs, or leftover meat—and I'll find a delicious Algerian recipe for us to make. *Wesh 'andak?* (What do you have?)", 'extras': {'signature': 'ErIJCq8JAWkUfRPNt6IVbf7x9KI1L5ac0Nbt+VoNbPXwJzZR3TjM+wwkhlgLSwJ6KcRKcHyPCPShxQS/g5y8Cd7QuU5P9dVgPkfGI5qOjXmwJzOpsUJZ8OR4ugoYEyrl8DS4bfU/k9YX8qgRuRXMf0Cr/dP/Jp2qgyrlyZyuu4FmMT9SBsSS71Gm5dIG1frTFnQ/7v04UeoWLrz4o5gdtmDAkV6N6LG8QaIOypTygqEUnU4sYcU4uMx0vWshEGWVc8wKdnK37QLjZDq8q3YdNW8OUE6LXV6+vUk9X3MmyZZB+PwAibbMVf6yU5P9jQamy2C+F+g80mfNmq8peKWxtRUAS6Gf7Yi6U3JZy11Ho3VGfMdSmTRosajyAHGZV7oor6RqC6XZJAxBMk7RXp4wNNS/T7dJW1aHmG0ZQbvYVKUhwgzR6/rdEyhq6orNgFBKY8Lmrs7hPI3gtN18g8N4BRVok3pRukAwOEXNC8B6oxlKhD0N9FPM3A7DQVUrFWBxSosXkaETnWmQNmqcmhpUrrmHuL/iuS6cWhcoeSmPrJ02e14IyWBSA7KcStN27JxlDLc4KbC8OzE7kd7sIQk1fYIAEG/5lslwJwEkCmu7t95EqToNBMCscTxNtgrGA4JFIYEPq78lYEO03cXn7SJQDklXNoQ5v6oY

## 6 · Web search for Algerian dishes outside the catalogue (notebook `1_2`)

`search_recipe_instructions` uses `TavilyClient().search` and only returns real
source links, so the agent can cite what it did not know from training.

In [7]:
import json

from notebook.chef_service import search_recipe_instructions

payload = json.loads(
    search_recipe_instructions.invoke({"query": "chakhchoukha recipe traditional algerian"})
)
print(payload["answer"] or "(no summary)")
for source in payload["sources"][:3]:
    print("-", source["title"], "→", source["url"])

Chakhchoukha is a traditional Algerian dish made with semolina dough, tomato sauce, chicken, carrots, and chickpeas. It is often served with torn flatbread. The dish is rich in flavor and protein.
- Algerian Chakhchoukha Recipe - The Odehlicious → https://theodehlicious.com/algerian-chakhchoukha-recipe
- Chakhchoukha → https://en.wikipedia.org/wiki/Chakhchoukha
- Algerian Chakhchoukha with Veal – Tasty Traditional Recipe → https://www.youtube.com/watch?v=RUJsA477y98&vl=en-US


## 7 · Run the same agent the apps use

`get_graph()` is exactly what the Streamlit app uses (with in-memory memory),
and `graph` (module-level) is what LangGraph Studio loads from `langgraph.json`.

In [8]:
from notebook.chef_service import get_graph, onboarding_message

ui_graph = get_graph()
profile = onboarding_message(name="Fatima", dietary_preferences=["Vegetarian"], allergies="peanuts")
try:
    greeting = chef.invoke_with_retry(ui_graph.invoke,
        {"messages": [{"role": "user", "content": profile}]},
        conversation_config("notebook-ui"),
    )["messages"][-1].content
    print(greeting)
except Exception as _gem_exc:
    chef.handle_gemini_error(_gem_exc)


C:\Users\Kassouar Fatima\Downloads\courses\Personal Algerian Chef\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': "Salam Fatima! Marhaba bik! 🌸\n\nIt is wonderful to cook with you today. I've noted that you are **vegetarian** and strictly avoiding **peanuts**.\n\nTell me, what leftovers or ingredients do you have in your fridge and pantry today? Let's turn them into something delicious together!", 'extras': {'signature': 'EoUGCoIGAWkUfRPEf7Jyh/V+o7Vmap93CuZ4MCDk6Csq9kUJZREtxu0/ur50zA/mpLGsXsCyZptzUb/Swvp1rbWU7KfImfjaoK6X6TFsqJL129Jiv54CzPycbnhmy001owI6CbKNUsqxoXWmltiWjlgIK2X8ioLeRc/Ogn9ZH4zPKUATw2etKOhwgj5Db0d4RbT+ITwSJAvziQ4dbkLk9Hghi6QDyIX638VCs3EqFb3Er2FGKN9RzHaif9q+RqGF8gkM5niCmefv0hgFg/88azr5JBz9CsFJZvlnn9V578GdGLHchOgdmubtA3M1yovxMRJq8+txP9qp4rl855eIZjQ/aGGgaLl7QhYO3DKgOiGLR7HZdQ0TVZi7v7shUfbqWyae6hkIn2tNT16hwO7ZxG0EOUKXQxBmpADWLBgedSgszF0NBqaZYpLgWAeVTeAlIi01FbB8dq5grULDoOezQWhz942rRolUXDQYZkL4W3oAR6avxFzK4OVlkkZETPATueR6cTqA44BrbZcUmSdp56QoAvIYH1Q6vhmAvrD54GPyPbJuIL99nPtfd/x1sp0A0UyS0/OhaC6N9N9rtGs+JHsk+aLeJUILKUgxmD5cQXIi9w4xHKNRjFBInN33m1yy7dO0MKr6UW8KJlx1HDt+zM

## Summary of functions used (and where they live in `chef_service.py`)

| Function / object | Meaning | In the project |
| --- | --- | --- |
| `ChatGoogleGenerativeAI(model=..., temperature=...)` | the Gemini model | `notebook/chef_service.py` → `_create_model()` |
| `model.invoke(...)` → `response.text`, `usage_metadata` | one model call | UI streaming in `streamlit/app.py` |
| `@tool` | python function → agent tool | `match_leftover_recipes`, `search_recipe_instructions` |
| `tavily.TavilyClient().search(query)` | web search with links | `search_recipe_instructions` |
| `create_agent(model, tools, system_prompt, checkpointer)` | the agent graph | `build_chef_graph()` |
| `MemorySaver()` + `checkpointer=` | RAM checkpointer | `get_graph()` (Streamlit) |
| `{"configurable": {"thread_id": ...}}` | conversation label | `conversation_config()` |
| `graph.get_state(config)` | saved snapshot | `response_from_state()` fallback |
| `graph.stream(..., stream_mode="messages")` | token streaming | `run_chef_turn()` |


## 8 · Full test suite — run it right here (no terminal)

The same 82 offline tests shown in `docs/demo.gif`: the 
Algerian-only whitelist, per-dish reachability, EN/FR/AR ingredient aliases,
foreign-food rejection, gluten flags, plus the service, simulation,
structure, and Streamlit UI tests. It is 100 % offline — no API keys needed.


In [9]:
import subprocess
import sys

root = globals().get("PROJECT_ROOT")
if root is None:
    raise SystemExit("Run the Setup cell first (it defines PROJECT_ROOT).")

print("Running the full offline test suite...")
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--no-header", "-p", "no:cacheprovider"],
    cwd=root,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    timeout=300,
)
print(result.stdout)
if result.stderr:
    print("--- stderr (first 2000 chars) ---")
    print(result.stderr[:2000])
print("[exit code]", result.returncode)


Running the full offline test suite...
........................................................................ [ 87%]
..........                                                               [100%]

[exit code] 0
